In [2]:
from pathlib import Path
import json
import re
import pandas as pd
from IPython.display import display

# --- Config ---
cwd = Path.cwd().resolve()
if (cwd / "notebooks").is_dir():
    project_root = cwd
else:
    notebooks_dir = next((path for path in [cwd, *cwd.parents] if path.name == "notebooks"), None)
    if notebooks_dir is None:
        raise ValueError("Could not locate the project root from the notebook working directory.")
    project_root = notebooks_dir.parent

results_root = project_root / "results"
timestamp = "Path_2025-03-20_19-00-00"  # March 20 at 19:00
run_dir = results_root / timestamp


def extract_param_count(model_name: str) -> float:
    match = re.search(r"(\d+(?:\.\d+)?)([KMBT])(?=-|$)", model_name, re.IGNORECASE)
    if not match:
        return float("inf")

    value = float(match.group(1))
    unit = match.group(2).upper()
    multipliers = {"K": 1e3, "M": 1e6, "B": 1e9, "T": 1e12}
    return value * multipliers[unit]


# --- Collect metrics for each model ---
# Structure target:
# columns = MultiIndex [model, {with image, without image}]
# rows    = [structure adherence pct, task accuracy pct]
records = {}

for vendor_dir in sorted(run_dir.iterdir()):
    if not vendor_dir.is_dir():
        continue
    for model_dir in sorted(vendor_dir.iterdir()):
        if not model_dir.is_dir():
            continue

        model_name = model_dir.name
        with_path = model_dir / "with_image.json"
        without_path = model_dir / "without_image.json"

        if not with_path.exists() or not without_path.exists():
            continue

        with with_path.open("r", encoding="utf-8") as f:
            with_data = json.load(f)
        with without_path.open("r", encoding="utf-8") as f:
            without_data = json.load(f)

        records[model_name] = {
            ("with image", "structure adherence pct"): with_data["overall_summary"]["structure_adherence_pct"],
            ("with image", "task accuracy pct"): with_data["overall_summary"]["task_accuracy_pct"],
            ("without image", "structure adherence pct"): without_data["overall_summary"]["structure_adherence_pct"],
            ("without image", "task accuracy pct"): without_data["overall_summary"]["task_accuracy_pct"],
        }

if not records:
    raise ValueError(f"No valid model result pairs found in: {run_dir}")

# --- Build table ---
row_order = ["structure adherence pct", "task accuracy pct"]
col_order = ["with image", "without image"]

data = {}
for model_name, vals in records.items():
    data[(model_name, "with image")] = [
        vals[("with image", "structure adherence pct")],
        vals[("with image", "task accuracy pct")],
    ]
    data[(model_name, "without image")] = [
        vals[("without image", "structure adherence pct")],
        vals[("without image", "task accuracy pct")],
    ]

table = pd.DataFrame(data, index=row_order)

# Sort models by parameter count, then by name, and enforce subcolumn order.
models_sorted = sorted(records.keys(), key=lambda name: (extract_param_count(name), name.lower()))
table = table.reindex(columns=pd.MultiIndex.from_product([models_sorted, col_order]))

# --- Format and display ---
styled = (
    table.style
    .format("{:.1f}%")
    .set_caption(f"Model Results ({timestamp})")
    .set_table_styles([
        {"selector": "caption", "props": [("font-size", "14px"), ("font-weight", "bold")]},
        {"selector": "th", "props": [("text-align", "center")]},
        {"selector": "td", "props": [("text-align", "center")]},
    ])
)

display(styled)